# Weather API Research & Exploration

## Objective

Before building a data ingestion pipeline, we first need to understand the source data.

In this notebook, we will:

- Review the Open-Meteo Historical Weather API
- Understand the available weather variables
- Explore the API response structure
- Convert the response into a tabular format
- Perform basic data profiling
- Identify candidate columns for Snowflake ingestion

This notebook focuses only on exploration and schema design.

The actual ingestion process will be implemented in a separate notebook.

In [24]:
# ==========================================================
# Import Required Libraries
# ==========================================================

import requests
import pandas as pd
from datetime import date, timedelta

pd.set_option("display.max_columns", None)

## 1. Review API Documentation

Before writing any code, it is important to understand what the API provides.

Open-Meteo Historical Weather API Documentation:

https://open-meteo.com/en/docs/historical-weather-api

### Observation

The API returns only the weather variables that are explicitly requested.

Therefore, we first review the documentation and identify the weather attributes that may be useful for analytics and machine learning.

For documentation purposes, capture a screenshot of the "Hourly Weather Variables" section and store it in the project documentation folder.

![alt text](image.png)

In [25]:
# ==========================================================
# Configure Location and Date Range
# ==========================================================

# Hyderabad Coordinates
LATITUDE = 17.3850
LONGITUDE = 78.4867

# Small sample range for exploration
START_DATE = "2025-01-01"
END_DATE = "2025-01-03"

print("Latitude :", LATITUDE)
print("Longitude:", LONGITUDE)

Latitude : 17.385
Longitude: 78.4867


## 2. Build a Minimal API Request

When exploring a new API, it is usually a good practice to start with a very small request.

Instead of requesting many weather attributes, we will initially request only temperature data.

This helps us understand the response structure before expanding the request.

In [26]:
# ==========================================================
# Minimal API Request
# ==========================================================

URL = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={LATITUDE}"
    f"&longitude={LONGITUDE}"
    f"&start_date={START_DATE}"
    f"&end_date={END_DATE}"
    "&hourly=temperature_2m"
)

print(URL)

https://archive-api.open-meteo.com/v1/archive?latitude=17.385&longitude=78.4867&start_date=2025-01-01&end_date=2025-01-03&hourly=temperature_2m


In [27]:
# ==========================================================
# Execute API Request
# ==========================================================

response = requests.get(URL)

response.raise_for_status()

weather_json = response.json()

print("API request successful")

API request successful


## 3. Explore API Response Structure

Before extracting any data, we should understand the structure of the JSON response.

This helps us identify:

- Metadata fields
- Observation fields
- Nested objects
- Candidate attributes for ingestion

In [28]:
# Top-Level JSON Keys

weather_json.keys()

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly'])

In [33]:
# View Metadata

print("Latitude:", weather_json["latitude"])
print("Longitude:", weather_json["longitude"])
print("Timezone:", weather_json["timezone"])
print("Elevation:", weather_json["elevation"])

Latitude: 17.398945
Longitude: 78.457085
Timezone: GMT
Elevation: 505.0


In [34]:
# Explore Hourly Observation Section

weather_json["hourly"].keys()

dict_keys(['time', 'temperature_2m'])

## Discussion

The API response consists of two categories of information.

### Metadata

Metadata describes the dataset itself.

Examples:

- latitude
- longitude
- timezone
- elevation

### Weather Observations

Actual weather measurements are stored under:

- hourly

Since we only requested temperature data, the hourly section currently contains:

- time
- temperature_2m

## 4. Select Candidate Weather Variables

After reviewing the Open-Meteo documentation, we can identify additional weather variables that may be useful for analytics and machine learning.

The following variables will be evaluated in the next API request.

These variables were identified from the documentation, not from the previous API response.

In [35]:
# ==========================================================
# Candidate Weather Variables
# Identified from API Documentation
# ==========================================================

candidate_variables = [
    "temperature_2m",
    "relative_humidity_2m",
    "pressure_msl",
    "wind_speed_10m",
    "precipitation",
    "cloud_cover"
]

candidate_variables

['temperature_2m',
 'relative_humidity_2m',
 'pressure_msl',
 'wind_speed_10m',
 'precipitation',
 'cloud_cover']

## 5. Fetch Extended Weather Dataset

Now that we understand the response structure and have identified candidate weather variables, we can retrieve a larger historical dataset.

For demonstration purposes, we will retrieve approximately 60 days of hourly weather observations.

In [36]:
# ==========================================================
# Configure Historical Date Range
# ==========================================================

END_DATE = date.today()

START_DATE = END_DATE - timedelta(days=60)

print("Start Date:", START_DATE)
print("End Date  :", END_DATE)

Start Date: 2026-07-17
End Date  : 2026-09-15


In [37]:
# ==========================================================
# Build Extended API Request
# ==========================================================

URL = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={LATITUDE}"
    f"&longitude={LONGITUDE}"
    f"&start_date={START_DATE}"
    f"&end_date={END_DATE}"
    f"&hourly={','.join(candidate_variables)}"
)

response = requests.get(URL)

response.raise_for_status()

weather_json = response.json()

print("Historical weather data retrieved")

Historical weather data retrieved


In [38]:
# Verify Returned Weather Variables

weather_json["hourly"].keys()

dict_keys(['time', 'temperature_2m', 'relative_humidity_2m', 'pressure_msl', 'wind_speed_10m', 'precipitation', 'cloud_cover'])

## 6. Transform JSON into a DataFrame

Machine learning and analytics workflows generally operate on tabular data.

Therefore, we will convert the nested JSON response into a Pandas DataFrame.

In [39]:
# ==========================================================
# Convert JSON to DataFrame
# ==========================================================

weather_df = pd.DataFrame(weather_json["hourly"])

weather_df.head()

,time,temperature_2m,relative_humidity_2m,pressure_msl,wind_speed_10m,precipitation,cloud_cover
0,2026-07-17T00:00,24.7,78,1007.4,11.0,0.1,100
1,2026-07-17T01:00,24.5,80,1008.1,10.4,0.1,100
2,2026-07-17T02:00,25.0,79,1008.6,11.6,0.2,100
3,2026-07-17T03:00,25.4,79,1009.0,13.1,0.2,100
4,2026-07-17T04:00,25.4,81,1009.3,11.4,0.4,100


## 7. Data Profiling

Before designing a target schema, it is important to understand:

- Dataset size
- Data types
- Missing values
- Statistical distributions

This is often the first stage of exploratory data analysis (EDA).

In [40]:
# Dataset Dimensions

weather_df.shape

(1464, 7)

In [41]:
# Data Types

weather_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1464 entries, 0 to 1463
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   time                  1464 non-null   object 
 1   temperature_2m        1464 non-null   float64
 2   relative_humidity_2m  1464 non-null   int64  
 3   pressure_msl          1464 non-null   float64
 4   wind_speed_10m        1464 non-null   float64
 5   precipitation         1464 non-null   float64
 6   cloud_cover           1464 non-null   int64  
dtypes: float64(4), int64(2), object(1)
memory usage: 80.2+ KB


In [42]:
# Statistical Summary

weather_df.describe()

,temperature_2m,relative_humidity_2m,pressure_msl,wind_speed_10m,precipitation,cloud_cover
count,1464.000000,1464.000000,1464.000000,1464.000000,1464.000000,1464.000000
mean,26.292008,73.476776,1007.565301,12.504372,0.162158,84.394809
std,2.296739,12.886306,2.671822,4.265155,0.445240,26.258870
min,22.000000,44.000000,997.800000,1.400000,0.000000,0.000000
25%,24.300000,63.000000,1006.200000,9.600000,0.000000,82.000000
50%,25.800000,76.000000,1007.800000,12.800000,0.000000,98.000000
75%,28.100000,84.000000,1009.200000,15.425000,0.100000,100.000000
max,32.200000,98.000000,1014.300000,25.300000,7.400000,100.000000


In [43]:
# Missing Value Analysis

weather_df.isnull().sum()

time                    0
temperature_2m          0
relative_humidity_2m    0
pressure_msl            0
wind_speed_10m          0
precipitation           0
cloud_cover             0
dtype: int64

## Schema Design Discussion

Based on our exploration, the following categories of columns have been identified.

### Metadata Columns

- latitude
- longitude
- timezone
- elevation

### Weather Observation Columns

- time
- temperature_2m
- relative_humidity_2m
- pressure_msl
- wind_speed_10m
- precipitation
- cloud_cover

These columns will be used to design the target Snowflake table in the next notebook.

The next notebook focuses on:

- Snowflake connectivity
- Table creation
- Data ingestion
- Load validation